# Turn-to-river experiment

Enumerate every legal river for two fixed private hands, retain the river label and showdown result, and calculate exact heads-up equity. A turn has 48 board children; fixing two compatible two-card hands leaves 44.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from cards import hand_names, make_hand
from showdown import INCOMPATIBLE
from turn import turn_outcomes

## One complete experiment

The first hand has the nut-heart draw; the second has top set. The table keeps the information that aggregate equity discards: exactly which cards produce each result.

In [2]:
turn = make_hand(['2h', '7h', 'Tc', 'Jc'])
first = make_hand(['Ah', 'Kh'])
second = make_hand(['Js', 'Jd'])
result = turn_outcomes(turn, first, second)

print('Turn: ', ' '.join(hand_names(turn)))
print('First:', ' '.join(hand_names(first)))
print('Second:', ' '.join(hand_names(second)))
if result == INCOMPATIBLE:
    print('Outcome: incompatible (the turn and private hands overlap)')
else:
    print(f'W/L/T: {result.wins}/{result.losses}/{result.ties}')
    print(f'Exact equity: {result.equity} = {float(result.equity):.2%}')

Turn:  Tc Jc 2h 7h
First: Kh Ah
Second: Jd Js
W/L/T: 10/34/0
Exact equity: 5/22 = 22.73%


In [3]:
rivers = (
    pd.DataFrame([{'river': None, 'outcome': 'incompatible'}])
    if result == INCOMPATIBLE
    else pd.DataFrame(
        [{'river': item.river_name, 'outcome': item.outcome} for item in result.rivers]
    )
)
rivers

,river,outcome
0,2c,loss
1,3c,loss
2,4c,loss
3,5c,loss
4,6c,loss
5,7c,loss
6,8c,loss
7,9c,loss
8,Qc,win
9,Kc,loss


In [4]:
# Grouping makes the changing rivers easy to explain without losing labels.
if result == INCOMPATIBLE:
    print('No rivers: the turn and private hands are incompatible.')
else:
    display(rivers.groupby('outcome', sort=False)['river'].apply(lambda values: ' '.join(values)))

outcome
loss    2c 3c 4c 5c 6c 7c 8c 9c Kc Ac 2d 3d 4d 5d 6d 7...
win                         Qc Qd 3h 4h 5h 6h 8h 9h Qh Qs
Name: river, dtype: str

## Three matchup types

Run the same exact calculation for a made pair against a straight draw, competing draws, and an overpair against a set.

In [5]:
matchups = [
    ('made pair vs straight draw', ['2c', '7d', '9h', 'Ks'], ['Kc', 'Qc'], ['Jh', 'Th']),
    ('competing draws', ['2h', '7h', 'Tc', 'Jc'], ['Ah', 'Kh'], ['Qc', '9c']),
    ('overpair vs set', ['2c', '7d', '9h', 'Js'], ['Ac', 'Ad'], ['9c', '9d']),
]

summary = []
for label, board_cards, first_cards, second_cards in matchups:
    experiment = turn_outcomes(
        make_hand(board_cards), make_hand(first_cards), make_hand(second_cards)
    )
    row = {
        'matchup': label,
        'turn': ' '.join(board_cards),
        'first': ' '.join(first_cards),
        'second': ' '.join(second_cards),
    }
    if experiment == INCOMPATIBLE:
        row.update({'W': None, 'L': None, 'T': None, 'equity': 'incompatible'})
    else:
        row.update({
            'W': experiment.wins, 'L': experiment.losses, 'T': experiment.ties,
            'equity': f'{experiment.equity} ({float(experiment.equity):.2%})',
        })
    summary.append(row)

pd.DataFrame(summary)

,matchup,turn,first,second,W,L,T,equity
0,made pair vs straight draw,2c 7d 9h Ks,Kc Qc,Jh Th,37,7,0,37/44 (84.09%)
1,competing draws,2h 7h Tc Jc,Ah Kh,Qc 9c,29,15,0,29/44 (65.91%)
2,overpair vs set,2c 7d 9h Js,Ac Ad,9c 9d,2,42,0,1/22 (4.55%)


## From labelled rivers to matrices

`turn_river_kernels(turn)` lazily yields `(river, compatibility, dominance)` for all 48 board children. Each matrix uses the established 1,326-hand ordering. Keeping this lazy is important: callers can consume, aggregate or discard one child kernel at a time instead of retaining all 48 dense pairs.